# Qwen3.6-35B-A3B Dual-T4 Kaggle Host

This notebook runs **Qwen3.6-35B-A3B** (GGUF `UD-Q4_K_XL` + `mmproj-F16` vision projector) on Kaggle with **2x NVIDIA Tesla T4** GPUs (30GB total VRAM) using `llama-server` and `pyngrok`.

### Prerequisites:
1. **Accelerator**: `GPU T4 x2` (Settings panel on right)
2. **Internet**: `Internet on` (Phone verified)
3. **Add Data**:
   - `tahsinekajolasalami/qwen36-35b-a3b-gguf`
   - `tahsinekajolasalami/llama-cpp-bin`
4. **Secrets** (Add-ons → Secrets):
   - `LLM_API_KEY`: API authentication key (e.g., `sk-qwen36-secret`)
   - `NGROK_AUTHTOKEN`: Ngrok account authtoken


In [ ]:
# CELL 1 — Check Hardware & Dependencies
%pip -q install pyngrok requests

import os, sys, time, shutil, subprocess
from pathlib import Path
import requests

print('Python:', sys.version.split()[0])
gpu = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader'],
    capture_output=True, text=True
)
if gpu.returncode != 0:
    raise RuntimeError('No NVIDIA GPU detected. Set Settings → Accelerator → GPU T4 x2, then restart session.')
print(gpu.stdout.strip())
gpu_count = len([line for line in gpu.stdout.splitlines() if line.strip()])
if gpu_count < 2:
    print('WARNING: Two T4 GPUs recommended. Found:', gpu_count)
else:
    print('✅ Two T4 GPUs detected. Ready to proceed.')


In [ ]:
# CELL 2 — Load Secrets & Locate Attached Files
from kaggle_secrets import UserSecretsClient
from pathlib import Path

try:
    secret_client = UserSecretsClient()
    LLM_API_KEY = secret_client.get_secret('LLM_API_KEY')
    NGROK_AUTHTOKEN = secret_client.get_secret('NGROK_AUTHTOKEN')
except Exception as exc:
    raise RuntimeError(
        'Kaggle secrets not available. Open Add-ons → Secrets, create LLM_API_KEY and NGROK_AUTHTOKEN, '
        'and enable both for this notebook.'
    ) from exc

if not LLM_API_KEY or not NGROK_AUTHTOKEN:
    raise RuntimeError('A required secret is empty. Check secret names and values in Add-ons → Secrets.')

INPUT_ROOT = Path('/kaggle/input')
model_files = [p for p in INPUT_ROOT.rglob('*UD-Q4_K_XL.gguf') if p.is_file()]
mmproj_files = [p for p in INPUT_ROOT.rglob('*mmproj*F16*.gguf') if p.is_file()]
server_files = [p for p in INPUT_ROOT.rglob('llama-server') if p.is_file()]

print('Model files found:', [str(p) for p in model_files])
print('Vision projector files found:', [str(p) for p in mmproj_files])
print('llama-server binaries found:', [str(p) for p in server_files])

if len(model_files) != 1:
    raise RuntimeError('Expected exactly one Qwen3.6 UD-Q4_K_XL GGUF. Check attached model dataset.')
if len(mmproj_files) != 1:
    raise RuntimeError('Expected exactly one mmproj F16 GGUF. Check attached Qwen model dataset.')
if len(server_files) != 1:
    raise RuntimeError('Expected exactly one llama-server binary. Check attached llama-cpp-bin dataset.')

MODEL_PATH = model_files[0]
MMPROJ_PATH = mmproj_files[0]
LLAMA_BIN_DIR = server_files[0].parent
PORT = 8080

print('Model size (GiB):', round(MODEL_PATH.stat().st_size / (1024**3), 2))
print('✅ All required files located.')


In [ ]:
# CELL 3 — Prepare Executable Runtime (Optional / Helper)
import os, shutil
from pathlib import Path

INPUT_ROOT = Path('/kaggle/input')
RUNTIME = Path('/kaggle/working/llama-runtime')
RUNTIME.mkdir(parents=True, exist_ok=True)

server_bins = [p for p in INPUT_ROOT.rglob('llama-server') if p.is_file()]
if server_bins:
    src_dir = server_bins[0].parent
    print(f'Copying runtime files from {src_dir} to {RUNTIME}...')
    for item in src_dir.iterdir():
        dest = RUNTIME / item.name
        if item.is_dir():
            shutil.copytree(item, dest, dirs_exist_ok=True)
        else:
            shutil.copy2(item, dest)
    (RUNTIME / 'llama-server').chmod(0o755)
    print('✅ llama-server ready at:', RUNTIME / 'llama-server')
else:
    print('⚠️ llama-server binary not found in /kaggle/input. Verify llama-cpp-bin dataset is attached.')


In [ ]:
# CELL 4 — Start Detached llama-server (Fully Self-Contained Dual-GPU Split)
import os, subprocess, time, glob, sys, shutil
from pathlib import Path
from kaggle_secrets import UserSecretsClient

INPUT_ROOT = Path('/kaggle/input')
RUNTIME = Path('/kaggle/working/llama-runtime')
SERVER_BIN = RUNTIME / 'llama-server'

# 1. Ensure executable runtime binary exists (auto-copy & chmod if needed)
if not SERVER_BIN.exists():
    print('llama-server not found in runtime directory. Searching /kaggle/input...')
    server_candidates = [p for p in INPUT_ROOT.rglob('llama-server') if p.is_file()]
    if not server_candidates:
        raise FileNotFoundError(
            'Could not find llama-server in /kaggle/input! Please ensure the dataset tahsinekajolasalami/llama-cpp-bin is attached.'
        )
    src_bin = server_candidates[0]
    src_dir = src_bin.parent
    print(f'Found binary at: {src_bin}')
    RUNTIME.mkdir(parents=True, exist_ok=True)
    print(f'Copying runtime files from {src_dir} to {RUNTIME}...')
    for item in src_dir.iterdir():
        dest = RUNTIME / item.name
        if item.is_dir():
            shutil.copytree(item, dest, dirs_exist_ok=True)
        else:
            shutil.copy2(item, dest)
    SERVER_BIN.chmod(0o755)
    print('✅ llama-server copied and made executable.')
else:
    SERVER_BIN.chmod(0o755)

# 2. Auto-detect MODEL_PATH and MMPROJ_PATH if needed
if 'MODEL_PATH' not in globals() or not Path(str(MODEL_PATH)).exists():
    model_files = [p for p in INPUT_ROOT.rglob('*UD-Q4_K_XL.gguf') if p.is_file()]
    if not model_files:
        raise RuntimeError('Model file (*UD-Q4_K_XL.gguf) not found in /kaggle/input. Verify attached dataset.')
    MODEL_PATH = model_files[0]

if 'MMPROJ_PATH' not in globals() or not Path(str(MMPROJ_PATH)).exists():
    mmproj_files = [p for p in INPUT_ROOT.rglob('*mmproj*F16*.gguf') if p.is_file()]
    if not mmproj_files:
        raise RuntimeError('Vision projector (*mmproj*F16*.gguf) not found in /kaggle/input.')
    MMPROJ_PATH = mmproj_files[0]

if 'LLM_API_KEY' not in globals() or not LLM_API_KEY:
    try:
        LLM_API_KEY = UserSecretsClient().get_secret('LLM_API_KEY')
    except Exception:
        LLM_API_KEY = 'sk-qwen36-secret'

print(f'Model:     {MODEL_PATH}')
print(f'Projector: {MMPROJ_PATH}')

LOG = '/kaggle/working/llama-server.log'
PIDFILE = '/kaggle/working/llama-server.pid'

# Kill any previous server
subprocess.run(['pkill', '-9', '-f', 'llama-server'], capture_output=True)
time.sleep(2)

# Dynamic library path setup
env = os.environ.copy()
py_ver = f'python{sys.version_info.major}.{sys.version_info.minor}'
nvidia_libs = glob.glob(f'/usr/local/lib/{py_ver}/dist-packages/nvidia/*/lib')
paths = ['/usr/local/nvidia/lib64', str(RUNTIME), *nvidia_libs, '/usr/lib/x86_64-linux-gnu']
env['LD_LIBRARY_PATH'] = ':'.join([p for p in paths if os.path.exists(p)])

command = [
    str(SERVER_BIN),
    '--model', str(MODEL_PATH),
    '--mmproj', str(MMPROJ_PATH),
    '--alias', 'qwen36-agent',
    '--alias', 'qwen3.6-35b-a3b',
    '--host', '0.0.0.0',
    '--port', '8080',
    '--ctx-size', '8192',
    '--n-gpu-layers', '999',
    '--split-mode', 'layer',
    '--tensor-split', '1,1',
    '--parallel', '1',
    '--jinja',
    '--api-key', str(LLM_API_KEY),
    '--image-min-tokens', '1024',
]

log = open(LOG, 'w')
process = subprocess.Popen(
    command,
    env=env,
    stdout=log,
    stderr=subprocess.STDOUT,
    start_new_session=True,
    stdin=subprocess.DEVNULL,
    close_fds=True,
)

with open(PIDFILE, 'w') as with_pid:
    with_pid.write(str(process.pid))

print('✅ Started detached llama-server. PID:', process.pid)
print('📄 Log path:', LOG)


In [ ]:
# CELL 5 — Monitor Startup & VRAM Loading
import requests, time, subprocess

print("Waiting for Qwen3.6 to finish loading into Dual T4 VRAM...")

for i in range(50):
    try:
        r = requests.get("http://127.0.0.1:8080/health", timeout=5)
        if r.status_code == 200:
            print("\n🎉 SUCCESS: Model is READY!")
            print(r.text)
            break
        print(f"Loading... ({i+1}/50) HTTP {r.status_code}")
    except requests.RequestException:
        print(f"Waiting for port 8080... ({i+1}/50)")
    time.sleep(10)
else:
    print("❌ Model did not become ready within timeout.")

print("\n=== GPU Memory Allocation ===")
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.used,memory.total", "--format=csv"])


In [ ]:
# CELL 6 — Launch Ngrok Public Tunnel (Self-Installing)
import time, requests, subprocess, sys

try:
    from pyngrok import ngrok
except ImportError:
    print('pyngrok not found. Installing now...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyngrok'])
    from pyngrok import ngrok

from kaggle_secrets import UserSecretsClient

PORT = 8080
if 'NGROK_AUTHTOKEN' not in globals() or not NGROK_AUTHTOKEN:
    try:
        NGROK_AUTHTOKEN = UserSecretsClient().get_secret('NGROK_AUTHTOKEN')
    except Exception:
        pass

if 'LLM_API_KEY' not in globals() or not LLM_API_KEY:
    try:
        LLM_API_KEY = UserSecretsClient().get_secret('LLM_API_KEY')
    except Exception:
        LLM_API_KEY = 'sk-qwen36-secret'

ngrok.set_auth_token(NGROK_AUTHTOKEN)

# Close any previous tunnel
try:
    ngrok.kill()
except Exception:
    pass
time.sleep(1)

# Connect to port 8080
tunnel = ngrok.connect(addr=PORT, proto="http")
PUBLIC_URL = tunnel.public_url.rstrip("/")
API_BASE_URL = PUBLIC_URL + "/v1"

print("=" * 60)
print("🚀 NGROK TUNNEL READY!")
print("Public URL:    ", PUBLIC_URL)
print("API Base URL:  ", API_BASE_URL)
print("API Key:       ", LLM_API_KEY)
print("=" * 60)

# Verify public endpoint
r = requests.get(
    f"{API_BASE_URL}/models",
    headers={"Authorization": f"Bearer {LLM_API_KEY}"},
    timeout=30
)
if r.status_code == 200:
    print("✅ Public API verified! Loaded models:", [m["id"] for m in r.json().get("data", [])])
else:
    print("❌ Public verification failed:", r.status_code, r.text[:300])


In [ ]:
# CELL 7 — Keep-Alive Loop
from datetime import datetime
import time, requests

PORT = 8080
api_url = globals().get('API_BASE_URL', f'http://127.0.0.1:{PORT}/v1')
print('Qwen3.6 is active at:', api_url)
print('Started at:', datetime.now().strftime('%Y-%m-%d %H:%M:%S'))
print('Keep this cell running while using the model.')
print('Press the Stop/Interrupt button when you are finished.')

try:
    while True:
        time.sleep(300)
        try:
            check = requests.get(f'http://127.0.0.1:{PORT}/health', timeout=5)
            status = check.json().get('status', 'unknown') if check.ok else f'HTTP {check.status_code}'
        except Exception as exc:
            status = f'not reachable: {exc}'
        print(datetime.now().strftime('%H:%M:%S'), 'health:', status)
except KeyboardInterrupt:
    print('Keep-alive stopped. You may now stop the Kaggle session.')
